# MIMIC-IV Preprocessing
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Luwam Major Kefali**

**Hilina Fissha Woreta**




This notebook handles the data prep side of the project , taking raw MIMIC-IV hospital data and turning it into something a model can actually use. That means cleaning up messy columns, defining the outcome variable, pulling in relevant features, and doing the train/val/test split.


## Setup

Importing what we need 

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

random_seed = 42

mimic_dir = "/kaggle/input/datasets/hilinafissha16/mimic-iv"

files = os.listdir(mimic_dir)
print("files found:")
for f in files:
    print(" -", f)

## Loading the raw tables

MIMIC-IV is spread across multiple CSVs. We only pull the columns we actually need since some of these files are massive (labevents alone has 158 million rows).

Quick rundown of what each table has:
- **admissions** , one row per hospital visit, includes dates, race, insurance
- **patients** , one row per patient, has age and sex
- **diagnoses_icd** , all diagnosis codes recorded per visit
- **icustays** , records for patients who went to the ICU
- **labevents** , every lab result, hence the 158M rows

In [ ]:
admissions = pd.read_csv(
    f"{mimic_dir}/admissions.csv",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime",
             "deathtime", "admission_type", "insurance", "race",
             "hospital_expire_flag"],
    parse_dates=["admittime", "dischtime", "deathtime"]
)

patients = pd.read_csv(
    f"{mimic_dir}/patients.csv",
    usecols=["subject_id", "gender", "anchor_age", "anchor_year", "dod"],
    parse_dates=["dod"]
)

diagnoses = pd.read_csv(
    f"{mimic_dir}/diagnoses_icd.csv",
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"]
)

icustays = pd.read_csv(
    f"{mimic_dir}/icustays.csv",
    usecols=["subject_id", "hadm_id", "intime", "outtime", "los"]
)

labevents = pd.read_csv(
    f"{mimic_dir}/labevents.csv",
    usecols=["subject_id", "hadm_id", "itemid", "valuenum", "storetime"],
    parse_dates=["storetime"]
)

print("admissions:", len(admissions))
print("patients:", len(patients))
print("diagnoses:", len(diagnoses))
print("icustays:", len(icustays))
print("labevents:", len(labevents))

## Building the base table

Merging admissions with patients to get demographics alongside each visit. MIMIC doesn't store age directly , it gives you an anchor year and age so you have to calculate it. We also compute length of stay in days.

Filtering to adults (18+) with a valid discharge time. Any pediatric cases or records with obvious data issues get dropped here.

In [ ]:
base = admissions.merge(
    patients[["subject_id", "gender", "anchor_age", "anchor_year", "dod"]],
    on="subject_id",
    how="left"
)

base["admit_year"] = base["admittime"].dt.year
base["age"] = base["anchor_age"] + (base["admit_year"] - base["anchor_year"])
base["age"] = base["age"].clip(18, 100)

base["los_days"] = (base["dischtime"] - base["admittime"]).dt.total_seconds() / 86400
base["los_days"] = base["los_days"].clip(lower=0)

base = base[
    (base["age"] >= 18) &
    (base["dischtime"].notna()) &
    (base["los_days"] > 0)
].copy()

print("adult admissions after filtering:", len(base))
print("age range:", base["age"].min(), "-", base["age"].max())

## Cleaning the protected attributes

Race, insurance, and sex are central to this project, and they're stored as messy free text in MIMIC. There are a ton of variations for the same group , "BLACK/AFRICAN AMERICAN", "BLACK/AFRICAN", "BLACK/CAPE VERDEAN" all mean the same thing but show up as separate strings.

Mapping everything down to consistent categories. If we left the raw values in, the model would treat those as three different groups, which would completely break the fairness analysis.

In [ ]:
race_map = {
    "WHITE": "White",
    "WHITE - BRAZILIAN": "White",
    "WHITE - EASTERN EUROPEAN": "White",
    "WHITE - OTHER EUROPEAN": "White",
    "WHITE - RUSSIAN": "White",
    "PORTUGUESE": "White",
    "BLACK/AFRICAN AMERICAN": "Black/African American",
    "BLACK/AFRICAN": "Black/African American",
    "BLACK/CAPE VERDEAN": "Black/African American",
    "BLACK/CARIBBEAN ISLAND": "Black/African American",
    "HISPANIC OR LATINO": "Hispanic/Latino",
    "HISPANIC/LATINO - CENTRAL AMERICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - COLOMBIAN": "Hispanic/Latino",
    "HISPANIC/LATINO - CUBAN": "Hispanic/Latino",
    "HISPANIC/LATINO - DOMINICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - GUATEMALAN": "Hispanic/Latino",
    "HISPANIC/LATINO - HONDURAN": "Hispanic/Latino",
    "HISPANIC/LATINO - MEXICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - PUERTO RICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - SALVADORAN": "Hispanic/Latino",
    "SOUTH AMERICAN": "Hispanic/Latino",
    "ASIAN": "Asian",
    "ASIAN - ASIAN INDIAN": "Asian",
    "ASIAN - CHINESE": "Asian",
    "ASIAN - KOREAN": "Asian",
    "ASIAN - SOUTH EAST ASIAN": "Asian",
    "ASIAN - VIETNAMESE": "Asian",
    "AMERICAN INDIAN/ALASKA NATIVE": "Other/Unknown",
    "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER": "Other/Unknown",
    "MULTIPLE RACE/ETHNICITY": "Other/Unknown",
    "OTHER": "Other/Unknown",
    "PATIENT DECLINED TO ANSWER": "Other/Unknown",
    "UNABLE TO OBTAIN": "Other/Unknown",
    "UNKNOWN": "Other/Unknown",
}

base["race_clean"] = (
    base["race"]
    .str.strip()
    .str.upper()
    .map(race_map)
    .fillna("Other/Unknown")
)

def clean_insurance(val):
    if pd.isna(val):
        return "Other"
    v = str(val).strip().upper()
    if "MEDICARE" in v:
        return "Medicare"
    if "MEDICAID" in v:
        return "Medicaid"
    if "PRIVATE" in v or "BLUE CROSS" in v or "COMMERCIAL" in v:
        return "Private"
    if "SELF" in v:
        return "Self-Pay"
    return "Other"

base["insurance_clean"] = base["insurance"].apply(clean_insurance)
base["sex"] = base["gender"].map({"M": "Male", "F": "Female"}).fillna("Unknown")

print("race distribution:")
print(base["race_clean"].value_counts())
print("\ninsurance distribution:")
print(base["insurance_clean"].value_counts())
print("\nsex distribution:")
print(base["sex"].value_counts())

## Creating the readmission label

The outcome variable: did this patient come back within 30 days of discharge?

30 days is the standard clinical threshold , US hospitals get financially penalized for 30-day readmissions because it often means something went wrong (discharged too early, not enough follow-up, etc.). So it's not an arbitrary cutoff.

One edge case: patients who died in-hospital can't be readmitted, so we exclude them from the positive label.

In [ ]:
base = base.sort_values(["subject_id", "admittime"]).reset_index(drop=True)

base["next_admittime"] = base.groupby("subject_id")["admittime"].shift(-1)

base["days_to_readmit"] = (
    base["next_admittime"] - base["dischtime"]
).dt.total_seconds() / 86400

base["readmitted_30d"] = (
    (base["days_to_readmit"] >= 0) &
    (base["days_to_readmit"] <= 30) &
    (base["hospital_expire_flag"] == 0)
).astype(int)

base["readmitted_30d"] = base["readmitted_30d"].fillna(0).astype(int)

print("readmission rate:", round(base["readmitted_30d"].mean() * 100, 1), "%")
print("readmitted:", base["readmitted_30d"].sum())
print("not readmitted:", (base["readmitted_30d"] == 0).sum())

## Comorbidity flags

Each admission has a list of ICD diagnosis codes. We use those to create 16 binary flags , one per condition. So `cm_diabetes = 1` means diabetes was documented during that stay.

MIMIC-IV contains both ICD-9 (older admissions, pre-2015) and ICD-10 (newer) codes. The two systems use different code formats , ICD-9 is mostly numeric ("428" = heart failure) while ICD-10 is alphanumeric ("I50"). We map both using prefix lists based on the Quan et al. (2005) dual comorbidity mappings, and combine them per admission. Without the ICD-9 side, every older admission would silently get zero comorbidities.

These are conditions known to be associated with readmission risk: heart failure, kidney disease, depression, substance use disorders, etc. We also add a simple count of how many a patient has.


In [ ]:
comorbidity_map_icd10 = {
    "chf":           ["I50"],
    "arrhythmia":    ["I44", "I45", "I46", "I47", "I48", "I49"],
    "hypertension":  ["I10", "I11", "I12", "I13", "I15"],
    "cpd":           ["J40", "J41", "J42", "J43", "J44", "J45", "J46", "J47"],
    "diabetes":      ["E10", "E11", "E12", "E13", "E14"],
    "renal_failure": ["N17", "N18", "N19"],
    "liver_disease": ["K70", "K71", "K72", "K73", "K74"],
    "cancer":        ["C00", "C01", "C02", "C03", "C04", "C05", "C06",
                      "C07", "C08", "C09", "C10", "C11", "C12", "C13",
                      "C14", "C15", "C16", "C17", "C18", "C19", "C20"],
    "obesity":       ["E66"],
    "depression":    ["F32", "F33"],
    "anxiety":       ["F40", "F41"],
    "alcohol_abuse": ["F10"],
    "drug_abuse":    ["F11", "F12", "F13", "F14", "F15", "F16", "F18", "F19"],
    "psychosis":     ["F20", "F22", "F23", "F24", "F25"],
    "coagulopathy":  ["D65", "D66", "D67", "D68", "D69"],
    "aids":          ["B20", "B21", "B22", "B24"],
}

comorbidity_map_icd9 = {
    "chf":           ["428"],
    "arrhythmia":    ["426", "427"],
    "hypertension":  ["401", "402", "403", "404", "405"],
    "cpd":           ["490", "491", "492", "493", "494", "495", "496"],
    "diabetes":      ["250"],
    "renal_failure": ["584", "585", "586"],
    "liver_disease": ["570", "571", "572", "573"],
    "cancer":        ["140", "141", "142", "143", "144", "145", "146",
                      "147", "148", "149", "150", "151", "152", "153", "154"],
    "obesity":       ["278"],
    "depression":    ["2962", "2963", "311"],
    "anxiety":       ["3000"],
    "alcohol_abuse": ["303", "3050"],
    "drug_abuse":    ["304", "3052", "3053", "3054", "3055", "3056", "3057",
                      "3058", "3059"],
    "psychosis":     ["295", "297", "298"],
    "coagulopathy":  ["286", "287"],
    "aids":          ["042", "043", "044"],
}

diagnoses["icd_code"] = diagnoses["icd_code"].str.strip().str.upper()
icd10 = diagnoses[diagnoses["icd_version"] == 10]
icd9  = diagnoses[diagnoses["icd_version"] == 9]

print("ICD-10 diagnosis rows:", len(icd10))
print("ICD-9 diagnosis rows: ", len(icd9))

comorbidity_df = diagnoses[["hadm_id"]].drop_duplicates()

for flag in comorbidity_map_icd10:
    pat10 = "|".join([f"^{p}" for p in comorbidity_map_icd10[flag]])
    pat9  = "|".join([f"^{p}" for p in comorbidity_map_icd9[flag]])
    matched = set(icd10[icd10["icd_code"].str.match(pat10, na=False)]["hadm_id"])
    matched |= set(icd9[icd9["icd_code"].str.match(pat9, na=False)]["hadm_id"])
    comorbidity_df[f"cm_{flag}"] = comorbidity_df["hadm_id"].isin(matched).astype(int)

cm_cols = [c for c in comorbidity_df.columns if c.startswith("cm_")]
comorbidity_df["comorbidity_count"] = comorbidity_df[cm_cols].sum(axis=1)

print("\ncomorbidity flags created:", len(cm_cols))
print("\nprevalence of each condition:")
print(comorbidity_df[cm_cols].mean().round(3).sort_values(ascending=False))


## Lab features

Lab values are one of the best signals for how sick a patient actually is at discharge. We pick 9 standard tests , creatinine, hemoglobin, sodium, potassium, etc.

Since patients get labs drawn multiple times during a stay, we use the last value before discharge. That's the most recent picture of the patient right before they leave, which is what matters for predicting what happens next.

In [ ]:
lab_items = {
    50912: "creatinine",
    51006: "bun",
    51222: "hemoglobin",
    51301: "wbc",
    50931: "glucose",
    50971: "potassium",
    50983: "sodium",
    51265: "platelets",
    50882: "bicarbonate",
}

labs = labevents[labevents["itemid"].isin(lab_items.keys())].copy()

labs = labs.merge(
    base[["hadm_id", "dischtime"]],
    on="hadm_id",
    how="inner"
)

labs = labs[labs["storetime"] <= labs["dischtime"]]

labs = (
    labs.sort_values("storetime")
    .groupby(["hadm_id", "itemid"])["valuenum"]
    .last()
    .reset_index()
)

labs["lab_name"] = labs["itemid"].map(lab_items)

lab_features = labs.pivot(
    index="hadm_id",
    columns="lab_name",
    values="valuenum"
).reset_index()
lab_features.columns.name = None

print("lab features shape:", lab_features.shape)
print("\nmissing rate per lab:")
print(lab_features.drop(columns="hadm_id").isna().mean().round(3).sort_values(ascending=False))

### Who has missing lab results?

Not every patient gets every lab test during their stay. Before deciding what to do with those gaps, it's worth checking whether the missingness is random or whether it falls more heavily on certain groups.

This matters a lot for the fairness analysis. If minority patients are disproportionately missing labs, that's a signal about unequal access to testing and it affects who ends up in our final dataset.

In [ ]:
temp = base[["hadm_id", "race_clean", "insurance_clean"]].merge(
    lab_features, on="hadm_id", how="left"
)

temp["any_lab_missing"] = temp[["bicarbonate", "bun", "creatinine",
                                 "glucose", "hemoglobin", "platelets",
                                 "potassium", "sodium", "wbc"]].isna().any(axis=1)

missing_group = temp[temp["any_lab_missing"] == True]
complete_group = temp[temp["any_lab_missing"] == False]

print("patients with at least one missing lab:", len(missing_group))
print("patients with complete labs:", len(complete_group))

print("\nrace distribution - missing labs (%):")
print((missing_group["race_clean"].value_counts() / len(missing_group) * 100).round(1))

print("\nrace distribution - complete labs (%):")
print((complete_group["race_clean"].value_counts() / len(complete_group) * 100).round(1))

print("\ninsurance distribution - missing labs (%):")
print((missing_group["insurance_clean"].value_counts() / len(missing_group) * 100).round(1))

print("\ninsurance distribution - complete labs (%):")
print((complete_group["insurance_clean"].value_counts() / len(complete_group) * 100).round(1))

del temp

In [ ]:
lab_cols = ["bicarbonate", "bun", "creatinine", "glucose",
            "hemoglobin", "platelets", "potassium", "sodium", "wbc"]

rate_check = base[["hadm_id", "race_clean", "insurance_clean"]].merge(
    lab_features, on="hadm_id", how="left"
)
rate_check["any_lab_missing"] = rate_check[lab_cols].isna().any(axis=1)

rate_race = rate_check.groupby("race_clean")["any_lab_missing"].mean().round(3) * 100
rate_ins  = rate_check.groupby("insurance_clean")["any_lab_missing"].mean().round(3) * 100

print("missingness RATE by race (% of that group missing at least one lab):")
print(rate_race.sort_values(ascending=False).to_string())

print("\nmissingness RATE by insurance (% of that group missing at least one lab):")
print(rate_ins.sort_values(ascending=False).to_string())

### Filtering to complete lab records

The analysis above shows the missingness is not random. Hispanic, Black, and Asian patients are missing at least one lab 33-35% of the time versus 24% for White patients. For insurance, Medicare patients stand out with far fewer missing labs (16.5%) than Medicaid, Private, or Other (34-42%).

For the model, we keep only patients who had all 9 lab tests recorded. Every lab value in the final dataset is a real observed measurement, not an estimate. The tradeoff is losing about 27% of admissions, but the remaining ~400k is still a large and usable dataset, and the model's decisions will be based on actual clinical data.

In [ ]:
lab_cols = ["bicarbonate", "bun", "creatinine", "glucose",
            "hemoglobin", "platelets", "potassium", "sodium", "wbc"]

before = len(lab_features)
lab_features_complete = lab_features.dropna(subset=lab_cols).copy()
after = len(lab_features_complete)

print(f"admissions with complete labs: {after:,} (removed {before - after:,})")

temp = base[["hadm_id", "race_clean", "insurance_clean"]].merge(
    lab_features_complete[["hadm_id"]], on="hadm_id", how="inner"
)

print("\nrace distribution after filtering:")
print((temp["race_clean"].value_counts() / len(temp) * 100).round(1))

del temp

## ICU stay features

Only about 15% of patients end up in the ICU. For those who do, total ICU time and number of ICU stays per admission are both useful signals. Multiple ICU stays in one admission tell something different than a single routine visit.

In [ ]:
icu_features = icustays.groupby("hadm_id").agg(
    icu_los_total=("los", "sum"),
    n_icu_stays=("los", "count")
).reset_index()

print("admissions with icu stay:", len(icu_features))
print("average icu length of stay (days):", round(icu_features["icu_los_total"].mean(), 2))
print("patients with more than one icu stay:", (icu_features["n_icu_stays"] > 1).sum())

## Prior admissions in the last 12 months

One of the stronger predictors of readmission is just how many times the patient has already been admitted in the past year. Someone coming in for the fifth time is in a very different situation than a first-timer.

We do a self-join on the admissions table to count prior visits within a 365-day window.

In [ ]:
base_sorted = base[["subject_id", "hadm_id", "admittime"]].sort_values(
    ["subject_id", "admittime"]
)

prior = base_sorted.merge(
    base_sorted.rename(columns={
        "hadm_id": "prev_hadm_id",
        "admittime": "prev_admittime"
    }),
    on="subject_id",
    how="left"
)

prior = prior[
    (prior["prev_admittime"] < prior["admittime"]) &
    (prior["prev_admittime"] >= prior["admittime"] - pd.Timedelta(days=365))
]

prior_counts = prior.groupby("hadm_id")["prev_hadm_id"].count().reset_index()
prior_counts.columns = ["hadm_id", "prior_admissions_12m"]

print("admissions with at least one prior visit:", len(prior_counts))
print("average prior admissions:", round(prior_counts["prior_admissions_12m"].mean(), 2))
print("max prior admissions:", prior_counts["prior_admissions_12m"].max())

## Assembling the full feature table

Now we put everything together into one table,  demographics, comorbidities, complete lab values, ICU features, and prior admissions. One row per admission.

We also add two intersectional columns: `race_x_sex` and `race_x_insurance`. These let us check for bias that only shows up at the intersection of two protected attributes, for example whether Black women on Medicaid face compounded disadvantages that looking at race or insurance alone wouldn't catch.

In [ ]:
features = (
    base[[
        "subject_id", "hadm_id", "admittime", "dischtime",
        "age", "sex", "race_clean", "insurance_clean",
        "admission_type", "los_days", "hospital_expire_flag",
        "readmitted_30d"
    ]]
    .merge(comorbidity_df,        on="hadm_id", how="left")
    .merge(lab_features_complete,  on="hadm_id", how="inner")
    .merge(icu_features,           on="hadm_id", how="left")
    .merge(prior_counts,           on="hadm_id", how="left")
)

for col in cm_cols:
    features[col] = features[col].fillna(0).astype(int)

features["comorbidity_count"]    = features["comorbidity_count"].fillna(0).astype(int)
features["n_icu_stays"]          = features["n_icu_stays"].fillna(0).astype(int)
features["icu_los_total"]        = features["icu_los_total"].fillna(0)
features["prior_admissions_12m"] = features["prior_admissions_12m"].fillna(0).astype(int)

features["race_x_sex"]       = features["race_clean"] + "_" + features["sex"]
features["race_x_insurance"] = features["race_clean"] + "_" + features["insurance_clean"]

print("final feature table shape:", features.shape)
print("readmission rate:", round(features["readmitted_30d"].mean() * 100, 1), "%")
print("race distribution:")
print((features["race_clean"].value_counts() / len(features) * 100).round(1))
print("any missing values left:")
missing = features.isna().sum()
print(missing[missing > 0] if missing[missing > 0].any() else "none")

## Train / validation / test split

70% train, 15% val, 15% test , split at the **patient level**, not the admission level. All admissions belonging to one `subject_id` land in exactly one split. Without this, the same patient could appear in both train and test, and the model could partially memorize repeat patients instead of learning generalizable readmission patterns , especially since prior admission history is our strongest feature.

Grouped splitting can't stratify exactly, so we verify afterwards that the readmission rate and race distribution are balanced across splits. With ~180k patients, random grouping balances well.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_seed)
train_idx, temp_idx = next(gss1.split(features, groups=features["subject_id"]))
train = features.iloc[train_idx].copy()
temp  = features.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_seed)
val_idx, test_idx = next(gss2.split(temp, groups=temp["subject_id"]))
val  = temp.iloc[val_idx].copy()
test = temp.iloc[test_idx].copy()

s_tr, s_va, s_te = set(train["subject_id"]), set(val["subject_id"]), set(test["subject_id"])
assert not (s_tr & s_va) and not (s_tr & s_te) and not (s_va & s_te), "patient overlap between splits!"
print("patient overlap check passed")

print("\npatients  | train:", len(s_tr), "| val:", len(s_va), "| test:", len(s_te))
print("admissions| train:", len(train), "| val:", len(val), "| test:", len(test))

print("\nreadmission rate per split:")
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"  {name}: {df['readmitted_30d'].mean()*100:.1f} %")

print("\nrace distribution per split (%):")
race_check = pd.DataFrame({
    name: (df["race_clean"].value_counts(normalize=True) * 100).round(1)
    for name, df in [("train", train), ("val", val), ("test", test)]
})
print(race_check.to_string())


## Saving the outputs

Saving everything as Parquet. Much faster and more compact than CSV at this scale. These files go straight into Stage 2 (the XGBoost model).

In [ ]:
import os
os.makedirs("/kaggle/working/splits", exist_ok=True)

train.to_parquet("/kaggle/working/splits/train.parquet", index=False)
val.to_parquet("/kaggle/working/splits/val.parquet",     index=False)
test.to_parquet("/kaggle/working/splits/test.parquet",   index=False)
features.to_parquet("/kaggle/working/mimic_features.parquet", index=False)

print("files saved:")
print("  train.parquet  -", len(train), "rows")
print("  val.parquet    -", len(val), "rows")
print("  test.parquet   -", len(test), "rows")
print("  mimic_features.parquet -", len(features), "rows")

## Notes index

A lookup table connecting each admission to its discharge note ID. We take the latest note per admission since that's the most complete summary of the patient's stay before discharge.

In [ ]:
discharge = pd.read_csv(
    f"{mimic_dir}/discharge.csv",
    usecols=["subject_id", "hadm_id", "note_id", "charttime"],
    parse_dates=["charttime"]
)

notes_index = discharge[
    discharge["hadm_id"].isin(features["hadm_id"])
].copy()

notes_index = (
    notes_index
    .sort_values("charttime")
    .groupby("hadm_id")
    .last()
    .reset_index()
)

print("notes index shape:", notes_index.shape)
print("sample:")
print(notes_index.head())

In [ ]:
notes_index.to_parquet("/kaggle/working/mimic_notes_index.parquet", index=False)
print("mimic_notes_index.parquet saved")
print("  rows:", len(notes_index))
print("  columns:", list(notes_index.columns))